# BCG Progression Prediction: Multimodal Case Study

This notebook demonstrates case-level progression-risk prediction using
previously trained survival models from the multimodal-survival-fusion
research repository.

We compare:

- Pathology-only
- RNA-only
- Clinical-only
- Concatenation fusion
- Gated fusion

We then examine how predicted risk changes when RNA is unavailable,
using models trained with RNA dropout.

The models were trained previously. In this notebook, we load the trained
models and apply them to selected patient cases.

## 1. Setup

We connect Google Drive and clone the research repository.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import os

%cd /content

if not os.path.exists("/content/multimodal-survival-fusion"):
    !git clone https://github.com/DIAGNijmegen/multimodal-survival-fusion.git

%cd /content/multimodal-survival-fusion
!pip install -e .

/content
Cloning into 'multimodal-survival-fusion'...
remote: Enumerating objects: 333, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (160/160), done.
remote: Total 333 (delta 155), reused 333 (delta 155), pack-reused 0 (from 0)
Receiving objects: 100% (333/333), 4.68 MiB | 33.72 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/multimodal-survival-fusion
Obtaining file:///content/multimodal-survival-fusion
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mm-survival (pyproject.toml) ... done
  Created wheel for mm-survival: filename=mm_survival-0.1.0-0.editable-py3-none-any.whl size=2181 sha256=572509bba25ab7bbea3dfa5ee0cb4dbc70c91117abc9b38c31e4e6289c6b1bab
  Stored in directory: /tmp/pip-ephem-wheel-cache-r689bw_i/wheels/3d/8d/75/926403b4f9e30c0d081e

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np

# Make repository source code directly importable
import sys
sys.path.insert(0, "/content/multimodal-survival-fusion/src")

# Project paths
DATA_ROOT = "/content/drive/MyDrive/Tutorial_data"
CHECKPOINT_ROOT = os.path.join(DATA_ROOT, "checkpoints")
CASE_DATA_ROOT = os.path.join(DATA_ROOT, "data")

Repository: /content/multimodal-survival-fusion
Data root: /content/drive/MyDrive/Tutorial_data
Checkpoint root: /content/drive/MyDrive/Tutorial_data/checkpoints
Number of cases: 10
Cases:
  3A_186 → Fold 3
  3B_418 → Fold 1
  3A_095 → Fold 2
  3A_031 → Fold 3
  3A_169 → Fold 2
  3A_143 → Fold 3
  3B_426 → Fold 1
  3A_142 → Fold 2
  3B_385 → Fold 2
  3A_041 → Fold 1


In [ ]:
# Data loading and preprocessing
from mm_survival.data.rna import (
    load_rna_matrix,
    transform_rna_with_medians,
)

from mm_survival.data.clinical import (
    load_clinical_table,
    transform_clinical_table,
)

from mm_survival.data.pathology import (
    load_pathology_feature_file,
    cap_tiles,
)

# Model architectures
from mm_survival.models.unimodal import (
    RNACoxModel,
    ClinicalCoxModel,
    PathologyCoxModel,
)

from mm_survival.models.multimodal_cox import (
    MultimodalCoxModel,
)

# RNA encoder used by the trained models
from mm_survival.models.encoders.rna.variance_filter import (
    VarianceFilterExtractor,
)

print("Research code imported successfully.")

## 2. Selected Cases

We selected 10 patient cases to examine how multimodal risk predictions
change across fusion strategies and under missing-RNA conditions.

In [ ]:
CASES = {
    "3A_186": 3,
    "3B_418": 1,
    "3A_095": 2,
    "3A_031": 3,
    "3A_169": 2,
    "3A_143": 3,
    "3B_426": 1,
    "3A_142": 2,
    "3B_385": 2,
    "3A_041": 1,
}

### Selected Case Characteristics

The selected cases represent different RNA-based molecular profiles and observed progression outcomes.

| Case | RNA characteristics | Expected effect | Outcome |
|---|---|---|---|
| **3A_186** | Strong basal and high relative FGFR3 expression | Increase risk | 🔴 Progression, 14 mo |
| **3B_418** | Cohort-high proliferation signature | Increase risk | 🔴 Progression, 12 mo |
| **3A_095** | Strong EMT, very low luminal expression | Increase risk | 🔴 Progression, 6 mo |
| **3A_031** | Highest relative FGFR3; low immune activity | Decrease risk | 🟢 No progression, 119 mo |
| **3A_169** | Low immune, EMT and proliferation signals | Decrease risk | 🟢 No progression, 155 mo |
| **3A_143** | Low EMT profile | Decrease risk | 🟢 No progression, 119 mo |
| **3B_426** | Immune-high and proliferative, consistent with concern | Reinforces decision | 🔴 Progression, 5 mo |
| **3A_142** | Low proliferation and basal activity | Reinforces lower risk | 🟢 No progression, 202 mo |
| **3B_385** | Strong aggressive RNA profile | Misleading increase | 🟢 No progression, 104 mo |
| **3A_041** | Lowest aggressive composite profile | Misleading decrease | 🔴 Progression, 10 mo |

## 3. Patient Data

For each selected case, we load the pathology, RNA, and clinical data.
Progression time and event status are obtained from the study labels.

In [ ]:
# Set up file paths for all selected patients

LABEL_PATH = os.path.join(
    CASE_DATA_ROOT, "labels", "labels.csv"
)

case_files = {}

for case_id in CASES:
    case_files[case_id] = {
        "Labels": LABEL_PATH,
        "Clinical": os.path.join(
            CASE_DATA_ROOT, "clinical", f"{case_id}_clinical.csv"
        ),
        "RNA": os.path.join(
            CASE_DATA_ROOT, "RNA", f"{case_id}_rna.csv"
        ),
        "WSI": os.path.join(
            CASE_DATA_ROOT, "WSI", f"{case_id}_HE.pt"
        ),
    }

# Check that the expected files exist
for case_id, files in case_files.items():
    print(f"\n{case_id} (Fold {CASES[case_id]}):")

    for name, path in files.items():
        print(f"  {name}: {os.path.exists(path)}")
        if not os.path.exists(path):
            print(f"    Missing: {path}")

In [ ]:
labels = pd.read_csv(LABEL_PATH)

print("Labels shape:", labels.shape)
print("Columns:", labels.columns.tolist())
print("\nFirst 5 rows:")
print(labels.head())

Labels shape: (10, 3)
Columns: ['sample_id', 'Time', 'Event']

First 5 rows:
  sample_id  Time  Event
0    3A_186    14      1
1    3B_418    12      1
2    3A_095     6      1
3    3A_031   119      0
4    3A_169   155      0


In [ ]:
# Create a label lookup for the selected cases

labels_by_case = labels.set_index("sample_id")[["Time", "Event"]]

print("Labels for selected cases:")
print(labels_by_case.loc[list(CASES.keys())])

Labels for selected cases:
           Time  Event
sample_id             
3A_186       14      1
3B_418       12      1
3A_095        6      1
3A_031      119      0
3A_169      155      0
3A_143      119      0
3B_426        5      1
3A_142      202      0
3B_385      104      0
3A_041       10      1


In [ ]:
# Load the three patient modalities for all selected cases

patient_data = {}
summary = []

for case_id in CASES:
    paths = case_files[case_id]

    wsi_features = load_pathology_feature_file(paths["WSI"])
    rna = load_rna_matrix(paths["RNA"])
    clinical = load_clinical_table(paths["Clinical"])

    patient_data[case_id] = {
        "WSI": wsi_features,
        "RNA": rna,
        "Clinical": clinical,
    }

    summary.append({
        "Patient": case_id,
        "Fold": CASES[case_id],
        "WSI features": str(wsi_features.shape),
        "RNA": str(rna.shape),
        "Clinical": str(clinical.shape),
    })

summary_df = pd.DataFrame(summary)

print("Patient data summary:")
display(summary_df)

Patient data summary:


,Patient,Fold,WSI features,RNA,Clinical
0,3A_186,3,"torch.Size([283962, 1024])","(1, 19359)","(1, 6)"
1,3B_418,1,"torch.Size([196813, 1024])","(1, 19359)","(1, 6)"
2,3A_095,2,"torch.Size([220657, 1024])","(1, 19359)","(1, 6)"
3,3A_031,3,"torch.Size([76536, 1024])","(1, 19359)","(1, 6)"
4,3A_169,2,"torch.Size([280848, 1024])","(1, 19359)","(1, 6)"
5,3A_143,3,"torch.Size([223902, 1024])","(1, 19359)","(1, 6)"
6,3B_426,1,"torch.Size([116382, 1024])","(1, 19359)","(1, 6)"
7,3A_142,2,"torch.Size([58557, 1024])","(1, 19359)","(1, 6)"
8,3B_385,2,"torch.Size([203724, 1024])","(1, 19359)","(1, 6)"
9,3A_041,1,"torch.Size([208217, 1024])","(1, 19359)","(1, 6)"


## 4. Load the Trained Models

The trained model checkpoints are stored in Google Drive.

We load the saved weights and experiment configurations for:
- Pathology-only
- RNA-only
- Clinical-only
- Concatenation fusion
- Gated fusion

The model architectures are provided by the research GitHub repository.
No model training is performed.

In [ ]:
# Checkpoint root
CHECKPOINT_ROOT = os.path.join(DATA_ROOT, "checkpoints")

checkpoint_paths = {}

for fold in [1, 2, 3]:
    checkpoint_paths[fold] = {
        "Pathology": os.path.join(
            CHECKPOINT_ROOT, f"fold_{fold}", "pathology_only", "model.pt"
        ),
        "RNA": os.path.join(
            CHECKPOINT_ROOT, f"fold_{fold}", "RNA_only", "model.pt"
        ),
        "Clinical": os.path.join(
            CHECKPOINT_ROOT, f"fold_{fold}", "clinical_only", "model.pt"
        ),
        "Concat fusion": os.path.join(
            CHECKPOINT_ROOT, f"fold_{fold}", "Concat_fusion", "model.pt"
        ),
        "Gated fusion": os.path.join(
            CHECKPOINT_ROOT, f"fold_{fold}", "gated_fusion", "model.pt"
        ),
    }

for fold, models in checkpoint_paths.items():
    print(f"\nFold {fold}:")
    for name, path in models.items():
        print(f"  {name}: {os.path.exists(path)}")


Fold 1:
  Pathology: True
  RNA: True
  Clinical: True
  Concat fusion: True
  Gated fusion: True

Fold 2:
  Pathology: True
  RNA: True
  Clinical: True
  Concat fusion: True
  Gated fusion: True

Fold 3:
  Pathology: True
  RNA: True
  Clinical: True
  Concat fusion: True
  Gated fusion: True


In [ ]:
# Load all trained checkpoints onto CPU, organized by fold

model_checkpoints = {}

for fold in [1, 2, 3]:
    model_checkpoints[fold] = {}

    for model_name, checkpoint_path in checkpoint_paths[fold].items():
        model_checkpoints[fold][model_name] = torch.load(
            checkpoint_path,
            map_location=torch.device("cpu"),
            weights_only=False
        )

print("Checkpoints loaded successfully.")

for fold in model_checkpoints:
    print(f"Fold {fold}:")
    for model_name in model_checkpoints[fold]:
        print(f"  ✓ {model_name}")

Checkpoints loaded successfully.
Fold 1:
  ✓ Pathology
  ✓ RNA
  ✓ Clinical
  ✓ Concat fusion
  ✓ Gated fusion
Fold 2:
  ✓ Pathology
  ✓ RNA
  ✓ Clinical
  ✓ Concat fusion
  ✓ Gated fusion
Fold 3:
  ✓ Pathology
  ✓ RNA
  ✓ Clinical
  ✓ Concat fusion
  ✓ Gated fusion


## 5. Load the Saved Preprocessing Information

The trained models were fitted using specific preprocessing parameters.

For case-level inference, we reuse these saved parameters rather than
fitting preprocessing again on the new patient.

This includes:
- Clinical preprocessing metadata
- RNA imputation medians
- RNA gene selection stored in the trained checkpoint

In [ ]:
# Load fold-specific preprocessing information for the fusion models

fusion_artifacts = {}

for fold in [1, 2, 3]:

    concat_dir = os.path.join(
        CHECKPOINT_ROOT, f"fold_{fold}", "Concat_fusion"
    )

    gated_dir = os.path.join(
        CHECKPOINT_ROOT, f"fold_{fold}", "gated_fusion"
    )

    fusion_artifacts[fold] = {}

    # -------------------------
    # Concatenation fusion
    # -------------------------

    concat_clinical_meta = json.load(
        open(os.path.join(concat_dir, "clinical_metadata.json"))
    )

    concat_rna_medians = pd.Series(
        json.load(
            open(os.path.join(concat_dir, "rna_imputation_medians.json"))
        )
    )

    concat_selected_genes = model_checkpoints[fold]["Concat fusion"][
        "selected_genes"
    ]

    fusion_artifacts[fold]["Concat fusion"] = {
        "clinical_meta": concat_clinical_meta,
        "rna_medians": concat_rna_medians,
        "selected_genes": concat_selected_genes,
    }

    # -------------------------
    # Gated fusion
    # -------------------------

    gated_clinical_meta = json.load(
        open(os.path.join(gated_dir, "clinical_metadata.json"))
    )

    gated_rna_medians = pd.Series(
        json.load(
            open(os.path.join(gated_dir, "rna_imputation_medians.json"))
        )
    )

    gated_selected_genes = model_checkpoints[fold]["Gated fusion"][
        "selected_genes"
    ]

    fusion_artifacts[fold]["Gated fusion"] = {
        "clinical_meta": gated_clinical_meta,
        "rna_medians": gated_rna_medians,
        "selected_genes": gated_selected_genes,
    }


# Print a clean summary
summary = []

for fold in [1, 2, 3]:
    for model_name in ["Concat fusion", "Gated fusion"]:

        artifacts = fusion_artifacts[fold][model_name]

        summary.append({
            "Fold": fold,
            "Model": model_name,
            "Clinical features": len(
                artifacts["clinical_meta"]["kept_features"]
            ),
            "RNA medians": len(
                artifacts["rna_medians"]
            ),
            "Selected genes": len(
                artifacts["selected_genes"]
            ),
        })

display(pd.DataFrame(summary))

,Fold,Model,Clinical features,RNA medians,Selected genes
0,1,Concat fusion,9,19359,2000
1,1,Gated fusion,9,19359,2000
2,2,Concat fusion,9,19359,2000
3,2,Gated fusion,9,19359,2000
4,3,Concat fusion,9,19359,2000
5,3,Gated fusion,9,19359,2000


## 6. Restore the Trained Models

The model architectures are provided by the research repository, while the
learned parameters are loaded from the saved checkpoints.

The RNA extractor also restores the gene-selection indices learned during
training.

No model is trained or fitted during this step.

In [ ]:
# Use CPU because the current Colab runtime does not have CUDA

device = torch.device("cpu")

print("Using device:", device)

Using device: cpu


###6A. RNA model

In [ ]:
def restore_rna_model(fold, input_dim):
    """Restore the trained RNA-only model for a specific fold."""

    checkpoint = model_checkpoints[fold]["RNA"]

    # Get configuration saved during training
    rna_config = checkpoint["experiment_config"]["model"]["rna"]
    rna_head_config = checkpoint["experiment_config"]["model"]["head"]

    # Create the RNA extractor
    rna_extractor = VarianceFilterExtractor(
        input_dim=input_dim,
        config=rna_config
    )

    # Restore learned gene-selection indices
    selected_idx = checkpoint["model_state"][
        "rna_extractor.selected_idx"
    ]

    rna_extractor._actual_k = len(selected_idx)
    rna_extractor.selected_idx = selected_idx.clone()

    # Recreate the trained MLP
    rna_extractor._mlp = rna_extractor._build_mlp(
        rna_extractor._actual_k
    )

    # Mark extractor as already fitted
    rna_extractor._is_fitted = True

    # Recreate the Cox model
    rna_model = RNACoxModel(
        rna_extractor=rna_extractor,
        head_hidden_dims=rna_head_config.get(
            "hidden_dims", [256]
        ),
        head_dropout=rna_head_config.get(
            "dropout", 0.30
        ),
        head_activation=rna_head_config.get(
            "activation", "selu"
        ),
    )

    # Load trained weights
    load_result = rna_model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    rna_model = rna_model.to(device)
    rna_model.eval()

    return rna_model, load_result

In [ ]:
# Restore RNA models for all folds

rna_models = {}

for fold in [1, 2, 3]:

    # Use any patient from this fold to get the RNA input dimension
    example_case = next(
        case_id for case_id, case_fold in CASES.items()
        if case_fold == fold
    )

    input_dim = patient_data[example_case]["RNA"].shape[1]

    rna_models[fold], load_result = restore_rna_model(
        fold=fold,
        input_dim=input_dim
    )

    print(f"Fold {fold} RNA model restored.")
    print("  Selected genes:", rna_models[fold].rna_extractor._actual_k)
    print("  Missing keys:", load_result.missing_keys)
    print("  Unexpected keys:", load_result.unexpected_keys)

Fold 1 RNA model restored.
  Selected genes: 2000
  Missing keys: []
  Unexpected keys: []
Fold 2 RNA model restored.
  Selected genes: 2000
  Missing keys: []
  Unexpected keys: []
Fold 3 RNA model restored.
  Selected genes: 2000
  Missing keys: []
  Unexpected keys: []


###6B. Clinical model

In [ ]:
def restore_clinical_model(fold):
    """Restore the trained clinical-only model for a specific fold."""

    checkpoint = model_checkpoints[fold]["Clinical"]

    # Get configuration saved during training
    clinical_config = checkpoint["experiment_config"]["model"]

    # Get the number of clinical features used during training
    clinical_dim = len(
        checkpoint["metadata"]["clinical_preprocessing"]["kept_features"]
    )

    # Recreate the clinical model
    clinical_model = ClinicalCoxModel(
        clinical_source="tabular",
        clinical_dim=clinical_dim,
        clinical_hidden_dims=clinical_config.get(
            "hidden_dims", [128]
        ),
        clinical_emb_dim=clinical_config.get(
            "projection_dim", 128
        ),
        clinical_dropout=clinical_config.get(
            "dropout", 0.30
        ),
        clinical_activation=clinical_config.get(
            "activation", "selu"
        ),
        head_hidden_dims=[],
    )

    # Load trained weights
    load_result = clinical_model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    clinical_model = clinical_model.to(device)
    clinical_model.eval()

    return clinical_model, load_result

In [ ]:
# Restore clinical models for all folds

clinical_models = {}

for fold in [1, 2, 3]:

    clinical_models[fold], load_result = restore_clinical_model(
        fold
    )

    print(f"Fold {fold} clinical model restored.")
    print("  Missing keys:", load_result.missing_keys)
    print("  Unexpected keys:", load_result.unexpected_keys)

Fold 1 clinical model restored.
  Missing keys: []
  Unexpected keys: []
Fold 2 clinical model restored.
  Missing keys: []
  Unexpected keys: []
Fold 3 clinical model restored.
  Missing keys: []
  Unexpected keys: []


###6C. Pathology model

In [ ]:
def restore_pathology_model(fold, pathology_in_dim):
    """Restore the trained pathology-only model for a specific fold."""

    checkpoint = model_checkpoints[fold]["Pathology"]

    # Get configurations saved during training
    pathology_config = checkpoint["experiment_config"]["model"]["pathology"]
    pathology_head_config = checkpoint["experiment_config"]["model"]["head"]

    # Recreate the pathology model
    pathology_model = PathologyCoxModel(
        pathology_in_dim=pathology_in_dim,
        pathology_emb_dim=pathology_config.get(
            "emb_dim", 256
        ),
        pathology_aggregator=pathology_config.get(
            "aggregator", "gated"
        ),
        pathology_attn_dim=pathology_config.get(
            "attn_dim", 128
        ),
        pathology_dropout=pathology_config.get(
            "dropout", 0.20
        ),
        head_hidden_dims=pathology_head_config.get(
            "hidden_dims", [256]
        ),
        head_dropout=pathology_head_config.get(
            "dropout", 0.30
        ),
        head_activation=pathology_head_config.get(
            "activation", "selu"
        ),
    )

    # Load trained weights
    load_result = pathology_model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    pathology_model = pathology_model.to(device)
    pathology_model.eval()

    return pathology_model, load_result

In [ ]:
# Restore pathology models for all folds

pathology_models = {}

for fold in [1, 2, 3]:

    # Get pathology input dimension from any patient in this fold
    example_case = next(
        case_id for case_id, case_fold in CASES.items()
        if case_fold == fold
    )

    pathology_in_dim = patient_data[example_case]["WSI"].shape[1]

    pathology_models[fold], load_result = restore_pathology_model(
        fold=fold,
        pathology_in_dim=pathology_in_dim
    )

    print(f"Fold {fold} pathology model restored.")
    print("  Missing keys:", load_result.missing_keys)
    print("  Unexpected keys:", load_result.unexpected_keys)

Fold 1 pathology model restored.
  Missing keys: []
  Unexpected keys: []
Fold 2 pathology model restored.
  Missing keys: []
  Unexpected keys: []
Fold 3 pathology model restored.
  Missing keys: []
  Unexpected keys: []


###6D. Multimodal models

In [ ]:
def restore_fusion_model(fold, model_name, rna_input_dim, pathology_in_dim):
    """Restore a trained multimodal model for a specific fold."""

    checkpoint = model_checkpoints[fold][model_name]

    model_cfg = checkpoint["experiment_config"]["model"]

    rna_cfg = model_cfg["rna"]
    clinical_cfg = model_cfg["clinical"]
    pathology_cfg = model_cfg["pathology"]
    head_cfg = model_cfg["head"]
    fusion_cfg = model_cfg["fusion"]

    # -------------------------
    # Restore RNA extractor
    # -------------------------
    extractor = VarianceFilterExtractor(
        input_dim=rna_input_dim,
        config=rna_cfg
    )

    selected_idx = checkpoint["model_state"][
        "rna_extractor.selected_idx"
    ]

    extractor._actual_k = len(selected_idx)
    extractor.selected_idx = selected_idx.clone()

    extractor._mlp = extractor._build_mlp(
        extractor._actual_k
    )

    extractor._is_fitted = True

    # -------------------------
    # Clinical dimension
    # -------------------------
    clinical_dim = len(
        fusion_artifacts[fold][model_name]["clinical_meta"]["kept_features"]
    )

    # -------------------------
    # Recreate multimodal model
    # -------------------------
    model = MultimodalCoxModel(
        rna_extractor=extractor,
        clinical_source="tabular",
        clinical_dim=clinical_dim,
        pathology_in_dim=pathology_in_dim,

        clinical_hidden_dims=clinical_cfg.get(
            "hidden_dims", [256, 128]
        ),
        clinical_emb_dim=clinical_cfg.get(
            "emb_dim", 128
        ),
        clinical_token_hidden_dim=clinical_cfg.get(
            "token_hidden_dim", 256
        ),
        clinical_token_out_dim=clinical_cfg.get(
            "token_out_dim", 128
        ),
        clinical_dropout=clinical_cfg.get(
            "dropout", 0.20
        ),
        clinical_activation=clinical_cfg.get(
            "activation", "selu"
        ),

        pathology_emb_dim=pathology_cfg.get(
            "emb_dim", 256
        ),
        pathology_aggregator=pathology_cfg.get(
            "aggregator", "gated"
        ),
        pathology_attn_dim=pathology_cfg.get(
            "attn_dim", 128
        ),
        pathology_dropout=pathology_cfg.get(
            "dropout", 0.20
        ),

        fusion_name=fusion_cfg["name"],

        head_hidden_dims=head_cfg.get(
            "hidden_dims", [256]
        ),
        head_dropout=head_cfg.get(
            "dropout", 0.30
        ),
        head_activation=head_cfg.get(
            "activation", "selu"
        ),
    )

    # -------------------------
    # Load trained weights
    # -------------------------
    load_result = model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    model = model.to(device)
    model.eval()

    return model, load_result

In [ ]:
# Restore Concat and Gated fusion models for all folds

fusion_models = {}

for fold in [1, 2, 3]:

    # Find a patient belonging to this fold
    example_case = next(
        case_id for case_id, case_fold in CASES.items()
        if case_fold == fold
    )

    rna_input_dim = patient_data[example_case]["RNA"].shape[1]
    pathology_in_dim = patient_data[example_case]["WSI"].shape[1]

    fusion_models[fold] = {}

    for model_name in ["Concat fusion", "Gated fusion"]:

        model, load_result = restore_fusion_model(
            fold=fold,
            model_name=model_name,
            rna_input_dim=rna_input_dim,
            pathology_in_dim=pathology_in_dim
        )

        fusion_models[fold][model_name] = model

        print(f"\nFold {fold} - {model_name}")
        print("  Missing keys:", load_result.missing_keys)
        print("  Unexpected keys:", load_result.unexpected_keys)


Fold 1 - Concat fusion
  Missing keys: []
  Unexpected keys: []

Fold 1 - Gated fusion
  Missing keys: []
  Unexpected keys: []

Fold 2 - Concat fusion
  Missing keys: []
  Unexpected keys: []

Fold 2 - Gated fusion
  Missing keys: []
  Unexpected keys: []

Fold 3 - Concat fusion
  Missing keys: []
  Unexpected keys: []

Fold 3 - Gated fusion
  Missing keys: []
  Unexpected keys: []


## 7. Preprocess the Case and Prepare Inputs

We now apply the same preprocessing used during training to the selected patient.

- RNA: apply the saved training-set imputation values
- Clinical: apply the saved clinical preprocessing metadata
- Pathology: apply the same tile-capping procedure

No preprocessing statistics are learned from the test case.

In [ ]:
# Preprocess all selected cases using their fold-specific training-time preprocessing

processed_data = {}

for case_id in CASES:

    fold = CASES[case_id]

    # Get patient data
    wsi_features = patient_data[case_id]["WSI"]
    rna = patient_data[case_id]["RNA"]
    clinical = patient_data[case_id]["Clinical"]

    processed_data[case_id] = {}

    # Process separately for each fusion model
    for model_name in ["Concat fusion", "Gated fusion"]:

        artifacts = fusion_artifacts[fold][model_name]

        # -------------------------
        # RNA
        # -------------------------
        rna_processed = transform_rna_with_medians(
            rna,
            artifacts["rna_medians"]
        )

        # -------------------------
        # Clinical
        # -------------------------
        clinical_processed = transform_clinical_table(
            clinical,
            artifacts["clinical_meta"]
        )

        # -------------------------
        # Pathology
        # -------------------------
        pathology_bag = cap_tiles(
            wsi_features,
            sample_id=case_id,
            seed=42,
            tile_cap=4096
        )

        # Convert to tensors
        rna_tensor = torch.tensor(
            rna_processed,
            dtype=torch.float32
        )

        clinical_tensor = torch.tensor(
            clinical_processed,
            dtype=torch.float32
        )

        processed_data[case_id][model_name] = {
            "RNA": rna_tensor,
            "Clinical": clinical_tensor,
            "Pathology": pathology_bag,
        }


# Clean summary
summary = []

for case_id in CASES:
    for model_name in ["Concat fusion", "Gated fusion"]:

        data = processed_data[case_id][model_name]

        summary.append({
            "Patient": case_id,
            "Fold": CASES[case_id],
            "Model": model_name,
            "RNA": str(data["RNA"].shape),
            "Clinical": str(data["Clinical"].shape),
            "Pathology": str(data["Pathology"].shape),
        })

display(pd.DataFrame(summary))

,Patient,Fold,Model,RNA,Clinical,Pathology
0,3A_186,3,Concat fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
1,3A_186,3,Gated fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
2,3B_418,1,Concat fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
3,3B_418,1,Gated fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
4,3A_095,2,Concat fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
5,3A_095,2,Gated fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
6,3A_031,3,Concat fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
7,3A_031,3,Gated fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
8,3A_169,2,Concat fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"
9,3A_169,2,Gated fusion,"torch.Size([1, 19359])","torch.Size([1, 9])","torch.Size([4096, 1024])"


### Verify the Prepared Inputs

The input dimensions should match the data expected by the trained models.

In [ ]:
# Summary of processed inputs for all selected cases

summary = []

for case_id in CASES:
    data = processed_data[case_id]["Gated fusion"]

    summary.append({
        "Patient": case_id,
        "Fold": CASES[case_id],
        "RNA features": data["RNA"].shape[1],
        "Clinical features": data["Clinical"].shape[1],
        "Pathology tiles": data["Pathology"].shape[0],
        "Pathology embedding dim": data["Pathology"].shape[1],
    })

display(pd.DataFrame(summary))

,Patient,Fold,RNA features,Clinical features,Pathology tiles,Pathology embedding dim
0,3A_186,3,19359,9,4096,1024
1,3B_418,1,19359,9,4096,1024
2,3A_095,2,19359,9,4096,1024
3,3A_031,3,19359,9,4096,1024
4,3A_169,2,19359,9,4096,1024
5,3A_143,3,19359,9,4096,1024
6,3B_426,1,19359,9,4096,1024
7,3A_142,2,19359,9,4096,1024
8,3B_385,2,19359,9,4096,1024
9,3A_041,1,19359,9,4096,1024


## 8. Predict Risk

We now use the restored trained models to predict a risk score for the selected patient.

The Cox model outputs a relative risk score. It is not a probability.
Higher values indicate higher predicted risk relative to other patients.

### 8.1 Pathology-only Model

The pathology-only model uses the patient's histopathology features to predict a Cox risk score.

In [ ]:
# Pathology-only risk prediction

pathology_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    model = pathology_models[fold]

    wsi_features = patient_data[case_id]["WSI"]

    pathology_bag = cap_tiles(
        wsi_features,
        sample_id=case_id,
        seed=42,
        tile_cap=4096
    )

    with torch.no_grad():
        risk = model([pathology_bag])

    pathology_risks[case_id] = risk.item()


pathology_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Pathology risk": pathology_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(pathology_results)

,Patient,Pathology risk,Time,Event
0,3A_186,0.672224,14,1
1,3B_418,-1.130377,12,1
2,3A_095,0.131498,6,1
3,3A_031,0.648959,119,0
4,3A_169,-0.762794,155,0
5,3A_143,0.624472,119,0
6,3B_426,-0.755841,5,1
7,3A_142,-0.286133,202,0
8,3B_385,0.925825,104,0
9,3A_041,-1.456040,10,1


### 8.2 RNA-only Model

The RNA-only model uses the patient's transcriptomic profile to predict a Cox risk score.

In [ ]:
# RNA-only risk prediction for all selected patients

rna_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    model = rna_models[fold]

    # Get this patient's RNA data
    rna = patient_data[case_id]["RNA"]

    # RNA preprocessing for RNA-only model
    # Use the same preprocessing that was used in your original
    # working RNA-only inference pipeline.
    rna_tensor = torch.tensor(
        rna.values,
        dtype=torch.float32
    )

    with torch.no_grad():
        risk = model(rna_tensor)

    rna_risks[case_id] = risk.item()


rna_results = pd.DataFrame([
    {
        "Patient": case_id,
        "RNA risk": rna_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(rna_results)

,Patient,RNA risk,Time,Event
0,3A_186,-0.486700,14,1
1,3B_418,0.800860,12,1
2,3A_095,0.429747,6,1
3,3A_031,-1.353258,119,0
4,3A_169,-0.565142,155,0
5,3A_143,-1.254008,119,0
6,3B_426,0.655895,5,1
7,3A_142,0.128075,202,0
8,3B_385,0.441780,104,0
9,3A_041,-0.132702,10,1


### 8.3 Clinical-only Model

The clinical-only model uses the patient's clinical features to predict a Cox risk score.

In [ ]:
# Clinical-only risk prediction for all selected patients

clinical_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    model = clinical_models[fold]

    # Get the fold-specific preprocessing metadata
    checkpoint = model_checkpoints[fold]["Clinical"]
    clinical_meta = checkpoint["metadata"]["clinical_preprocessing"]

    # Apply the same preprocessing used during training
    clinical_processed = transform_clinical_table(
        patient_data[case_id]["Clinical"],
        clinical_meta
    )

    clinical_tensor = torch.tensor(
        clinical_processed,
        dtype=torch.float32
    )

    with torch.no_grad():
        risk = model(clinical_tensor)

    clinical_risks[case_id] = risk.item()


clinical_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Clinical risk": clinical_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(clinical_results)

,Patient,Clinical risk,Time,Event
0,3A_186,0.427817,14,1
1,3B_418,0.984828,12,1
2,3A_095,0.688770,6,1
3,3A_031,-0.698413,119,0
4,3A_169,-0.099268,155,0
5,3A_143,-0.488468,119,0
6,3B_426,0.521662,5,1
7,3A_142,-0.200271,202,0
8,3B_385,0.255263,104,0
9,3A_041,1.257551,10,1


### 8.4 Concatenation Fusion

The concatenation model combines the pathology, RNA, and clinical embeddings by concatenating them before the survival prediction head.

In [ ]:
# Concatenation fusion risk prediction for all selected patients

concat_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    model = fusion_models[fold]["Concat fusion"]

    # Get the correctly preprocessed data for this fold/model
    data = processed_data[case_id]["Concat fusion"]

    rna_tensor = data["RNA"]
    clinical_tensor = data["Clinical"]
    pathology_bag = data["Pathology"]

    with torch.no_grad():
        risk = model(
            rna_tensor,
            clinical_tensor,
            [pathology_bag]
        )

    concat_risks[case_id] = risk.item()


concat_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Concat risk": concat_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(concat_results)

,Patient,Concat risk,Time,Event
0,3A_186,-0.862797,14,1
1,3B_418,-1.054877,12,1
2,3A_095,1.660013,6,1
3,3A_031,-2.531125,119,0
4,3A_169,-1.873462,155,0
5,3A_143,-2.636393,119,0
6,3B_426,-1.392895,5,1
7,3A_142,-0.639147,202,0
8,3B_385,0.644160,104,0
9,3A_041,-2.491306,10,1


### 8.5 Gated Fusion

The gated fusion model combines the three modality embeddings using a learned patient-specific gating mechanism.

In [ ]:
# Gated fusion risk prediction for all selected patients

gated_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    model = fusion_models[fold]["Gated fusion"]

    # Get correctly preprocessed data for this fold/model
    data = processed_data[case_id]["Gated fusion"]

    rna_tensor = data["RNA"]
    clinical_tensor = data["Clinical"]
    pathology_bag = data["Pathology"]

    with torch.no_grad():
        risk = model(
            rna_tensor,
            clinical_tensor,
            [pathology_bag]
        )

    gated_risks[case_id] = risk.item()


gated_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Gated risk": gated_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(gated_results)

,Patient,Gated risk,Time,Event
0,3A_186,-0.351553,14,1
1,3B_418,-0.516763,12,1
2,3A_095,0.122275,6,1
3,3A_031,-2.094043,119,0
4,3A_169,-1.629674,155,0
5,3A_143,-2.336698,119,0
6,3B_426,-1.096449,5,1
7,3A_142,-1.231116,202,0
8,3B_385,-0.247311,104,0
9,3A_041,-2.028022,10,1


### 8.6 Risk Score Summary

The predictions from all models are summarized below. Risk scores are relative Cox model outputs and should not be interpreted as probabilities.

In [ ]:
# Combine all model predictions

risk_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Pathology-only": pathology_risks[case_id],
        "RNA-only": rna_risks[case_id],
        "Clinical-only": clinical_risks[case_id],
        "Concatenation fusion": concat_risks[case_id],
        "Gated fusion": gated_risks[case_id],
        "Time": labels_by_case.loc[case_id, "Time"],
        "Event": labels_by_case.loc[case_id, "Event"],
    }
    for case_id in CASES
])

display(risk_results)

,Patient,Pathology-only,RNA-only,Clinical-only,Concatenation fusion,Gated fusion,Time,Event
0,3A_186,0.672224,-0.486700,0.427817,-0.862797,-0.351553,14,1
1,3B_418,-1.130377,0.800860,0.984828,-1.054877,-0.516763,12,1
2,3A_095,0.131498,0.429747,0.688770,1.660013,0.122275,6,1
3,3A_031,0.648959,-1.353258,-0.698413,-2.531125,-2.094043,119,0
4,3A_169,-0.762794,-0.565142,-0.099268,-1.873462,-1.629674,155,0
5,3A_143,0.624472,-1.254008,-0.488468,-2.636393,-2.336698,119,0
6,3B_426,-0.755841,0.655895,0.521662,-1.392895,-1.096449,5,1
7,3A_142,-0.286133,0.128075,-0.200271,-0.639147,-1.231116,202,0
8,3B_385,0.925825,0.441780,0.255263,0.644160,-0.247311,104,0
9,3A_041,-1.456040,-0.132702,1.257551,-2.491306,-2.028022,10,1


## 9. Missing RNA Inference

[TODO: Demonstrate prediction when RNA data are unavailable.]

- Remove/mask the RNA modality
- Use the trained model with missing RNA
- Compare the risk score with complete-modality inference
- Explain how the model handles the missing modality

In [ ]:
# Check missing-RNA checkpoint files

missing_rna_paths = {}

for fold in [1, 2, 3]:
    missing_rna_paths[fold] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:
        missing_rna_paths[fold][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            model_dir = os.path.join(
                CHECKPOINT_ROOT,
                f"fold_{fold}",
                dropout,
                model_name
            )

            missing_rna_paths[fold][dropout][model_name] = {
                "model": os.path.join(model_dir, "model.pt"),
                "clinical_metadata": os.path.join(
                    model_dir, "clinical_metadata.json"
                ),
                "rna_medians": os.path.join(
                    model_dir, "rna_imputation_medians.json"
                ),
                "selected_genes": os.path.join(
                    model_dir, "selected_genes.txt"
                ),
            }

In [ ]:
# Load missing-RNA checkpoints

missing_rna_checkpoints = {}

for fold in [1, 2, 3]:

    missing_rna_checkpoints[fold] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        missing_rna_checkpoints[fold][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            checkpoint_path = (
                missing_rna_paths[fold][dropout][model_name]["model"]
            )

            checkpoint = torch.load(
                checkpoint_path,
                map_location=torch.device("cpu"),
                weights_only=False
            )

            missing_rna_checkpoints[fold][dropout][model_name] = checkpoint

In [ ]:
def restore_missing_rna_model(
    fold,
    dropout,
    model_name,
    rna_input_dim,
    pathology_in_dim
):
    """Restore a missing-RNA-trained fusion model."""

    checkpoint = missing_rna_checkpoints[fold][dropout][model_name]

    model_cfg = checkpoint["experiment_config"]["model"]

    rna_cfg = model_cfg["rna"]
    clinical_cfg = model_cfg["clinical"]
    pathology_cfg = model_cfg["pathology"]
    head_cfg = model_cfg["head"]
    fusion_cfg = model_cfg["fusion"]

    # Restore RNA feature extractor
    extractor = VarianceFilterExtractor(
        input_dim=rna_input_dim,
        config=rna_cfg
    )

    selected_idx = checkpoint["model_state"][
        "rna_extractor.selected_idx"
    ]

    extractor._actual_k = len(selected_idx)
    extractor.selected_idx = selected_idx.clone()
    extractor._mlp = extractor._build_mlp(
        extractor._actual_k
    )
    extractor._is_fitted = True

    # Use missing-RNA-specific clinical preprocessing
    clinical_meta = missing_rna_artifacts[
        fold
    ][dropout][model_name]["clinical_meta"]

    clinical_dim = len(
        clinical_meta["kept_features"]
    )

    # Restore model
    model = MultimodalCoxModel(
        rna_extractor=extractor,
        clinical_source="tabular",
        clinical_dim=clinical_dim,
        pathology_in_dim=pathology_in_dim,

        clinical_hidden_dims=clinical_cfg.get(
            "hidden_dims", [256, 128]
        ),
        clinical_emb_dim=clinical_cfg.get(
            "emb_dim", 128
        ),
        clinical_token_hidden_dim=clinical_cfg.get(
            "token_hidden_dim", 256
        ),
        clinical_token_out_dim=clinical_cfg.get(
            "token_out_dim", 128
        ),
        clinical_dropout=clinical_cfg.get(
            "dropout", 0.20
        ),
        clinical_activation=clinical_cfg.get(
            "activation", "selu"
        ),

        pathology_emb_dim=pathology_cfg.get(
            "emb_dim", 256
        ),
        pathology_aggregator=pathology_cfg.get(
            "aggregator", "gated"
        ),
        pathology_attn_dim=pathology_cfg.get(
            "attn_dim", 128
        ),
        pathology_dropout=pathology_cfg.get(
            "dropout", 0.20
        ),

        fusion_name=fusion_cfg["name"],

        head_hidden_dims=head_cfg.get(
            "hidden_dims", [256]
        ),
        head_dropout=head_cfg.get(
            "dropout", 0.30
        ),
        head_activation=head_cfg.get(
            "activation", "selu"
        ),
    )

    model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    model = model.to(device)
    model.eval()

    return model

In [ ]:
missing_rna_models = {}

for fold in [1, 2, 3]:

    missing_rna_models[fold] = {}

    example_case = next(
        case_id for case_id, case_fold in CASES.items()
        if case_fold == fold
    )

    rna_input_dim = patient_data[example_case]["RNA"].shape[1]
    pathology_in_dim = patient_data[example_case]["WSI"].shape[1]

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        missing_rna_models[fold][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            model = restore_missing_rna_model(
                fold=fold,
                dropout=dropout,
                model_name=model_name,
                rna_input_dim=rna_input_dim,
                pathology_in_dim=pathology_in_dim
            )

            missing_rna_models[fold][dropout][model_name] = model

In [ ]:
# Load missing-RNA preprocessing artifacts

missing_rna_artifacts = {}

for fold in [1, 2, 3]:
    missing_rna_artifacts[fold] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:
        missing_rna_artifacts[fold][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            paths = missing_rna_paths[fold][dropout][model_name]

            clinical_meta = json.load(
                open(paths["clinical_metadata"])
            )

            rna_medians = pd.Series(
                json.load(open(paths["rna_medians"]))
            )

            with open(paths["selected_genes"], "r") as f:
                selected_genes = [
                    line.strip()
                    for line in f
                    if line.strip()
                ]

            missing_rna_artifacts[fold][dropout][model_name] = {
                "clinical_meta": clinical_meta,
                "rna_medians": rna_medians,
                "selected_genes": selected_genes,
            }

In [ ]:
def restore_missing_rna_model(
    fold,
    dropout,
    model_name,
    rna_input_dim,
    pathology_in_dim
):
    checkpoint = missing_rna_checkpoints[fold][dropout][model_name]

    model_cfg = checkpoint["experiment_config"]["model"]
    rna_cfg = model_cfg["rna"]
    clinical_cfg = model_cfg["clinical"]
    pathology_cfg = model_cfg["pathology"]
    head_cfg = model_cfg["head"]
    fusion_cfg = model_cfg["fusion"]

    # Restore RNA feature extractor
    extractor = VarianceFilterExtractor(
        input_dim=rna_input_dim,
        config=rna_cfg
    )

    selected_idx = checkpoint["model_state"][
        "rna_extractor.selected_idx"
    ]

    extractor._actual_k = len(selected_idx)
    extractor.selected_idx = selected_idx.clone()
    extractor._mlp = extractor._build_mlp(
        extractor._actual_k
    )
    extractor._is_fitted = True

    # Use missing-RNA-specific clinical preprocessing
    clinical_meta = missing_rna_artifacts[
        fold
    ][dropout][model_name]["clinical_meta"]

    clinical_dim = len(
        clinical_meta["kept_features"]
    )

    # Restore model
    model = MultimodalCoxModel(
        rna_extractor=extractor,
        clinical_source="tabular",
        clinical_dim=clinical_dim,
        pathology_in_dim=pathology_in_dim,

        clinical_hidden_dims=clinical_cfg.get(
            "hidden_dims", [256, 128]
        ),
        clinical_emb_dim=clinical_cfg.get(
            "emb_dim", 128
        ),
        clinical_token_hidden_dim=clinical_cfg.get(
            "token_hidden_dim", 256
        ),
        clinical_token_out_dim=clinical_cfg.get(
            "token_out_dim", 128
        ),
        clinical_dropout=clinical_cfg.get(
            "dropout", 0.20
        ),
        clinical_activation=clinical_cfg.get(
            "activation", "selu"
        ),

        pathology_emb_dim=pathology_cfg.get(
            "emb_dim", 256
        ),
        pathology_aggregator=pathology_cfg.get(
            "aggregator", "gated"
        ),
        pathology_attn_dim=pathology_cfg.get(
            "attn_dim", 128
        ),
        pathology_dropout=pathology_cfg.get(
            "dropout", 0.20
        ),

        fusion_name=fusion_cfg["name"],

        head_hidden_dims=head_cfg.get(
            "hidden_dims", [256]
        ),
        head_dropout=head_cfg.get(
            "dropout", 0.30
        ),
        head_activation=head_cfg.get(
            "activation", "selu"
        ),
    )

    model.load_state_dict(
        checkpoint["model_state"],
        strict=False
    )

    model = model.to(device)
    model.eval()

    return model

In [ ]:
missing_rna_models = {}

for fold in [1, 2, 3]:

    missing_rna_models[fold] = {}

    example_case = next(
        case_id
        for case_id, case_fold in CASES.items()
        if case_fold == fold
    )

    rna_input_dim = patient_data[example_case]["RNA"].shape[1]
    pathology_in_dim = patient_data[example_case]["WSI"].shape[1]

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        missing_rna_models[fold][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            model = restore_missing_rna_model(
                fold=fold,
                dropout=dropout,
                model_name=model_name,
                rna_input_dim=rna_input_dim,
                pathology_in_dim=pathology_in_dim
            )

            missing_rna_models[fold][dropout][model_name] = model

In [ ]:
from mm_survival.training.rna_missing import (
    apply_rna_mask,
    missing_rna_mask_like,
)

In [ ]:
missing_rna_processed_data = {}

# Preprocess pathology and clinical data once per patient/model
for case_id in CASES:

    fold = CASES[case_id]

    rna = patient_data[case_id]["RNA"]
    clinical = patient_data[case_id]["Clinical"]
    wsi_features = patient_data[case_id]["WSI"]

    pathology_bag = cap_tiles(
        wsi_features,
        sample_id=case_id,
        seed=42,
        tile_cap=4096
    )

    missing_rna_processed_data[case_id] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        missing_rna_processed_data[case_id][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            artifacts = missing_rna_artifacts[
                fold
            ][dropout][model_name]

            rna_processed = transform_rna_with_medians(
                rna,
                artifacts["rna_medians"]
            )

            clinical_processed = transform_clinical_table(
                clinical,
                artifacts["clinical_meta"]
            )

            missing_rna_processed_data[
                case_id
            ][dropout][model_name] = {
                "RNA": torch.tensor(
                    rna_processed,
                    dtype=torch.float32
                ),
                "Clinical": torch.tensor(
                    clinical_processed,
                    dtype=torch.float32
                ),
                "Pathology": pathology_bag,
            }

In [ ]:
missing_rna_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    missing_rna_risks[case_id] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        missing_rna_risks[case_id][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            model = missing_rna_models[fold][dropout][model_name]
            data = missing_rna_processed_data[case_id][dropout][model_name]

            rna_tensor = data["RNA"]
            clinical_tensor = data["Clinical"]
            pathology_bag = data["Pathology"]

            # Mask RNA completely
            rna_mask = missing_rna_mask_like(rna_tensor)
            missing_rna_tensor = apply_rna_mask(
                rna_tensor,
                rna_mask
            )

            # Predict risk with RNA missing
            with torch.no_grad():
                risk = model.forward_all(
                    missing_rna_tensor,
                    clinical_tensor,
                    [pathology_bag],
                    device=device,
                    rna_mask=rna_mask
                )

            missing_rna_risks[case_id][dropout][model_name] = risk.item()

In [ ]:
complete_dropout_risks = {}

for case_id in CASES:

    fold = CASES[case_id]
    complete_dropout_risks[case_id] = {}

    for dropout in ["missing_rna_30", "missing_rna_50"]:

        complete_dropout_risks[case_id][dropout] = {}

        for model_name in ["Concat_fusion", "gated_fusion"]:

            model = missing_rna_models[fold][dropout][model_name]

            data = missing_rna_processed_data[
                case_id
            ][dropout][model_name]

            rna_tensor = data["RNA"]
            clinical_tensor = data["Clinical"]
            pathology_bag = data["Pathology"]

            # Complete-RNA inference
            with torch.no_grad():
                risk = model.forward_all(
                    rna_tensor,
                    clinical_tensor,
                    [pathology_bag],
                    device=device
                )

            complete_dropout_risks[
                case_id
            ][dropout][model_name] = risk.item()

In [ ]:
comparison_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Outcome": (
            "Progression"
            if labels_by_case.loc[case_id, "Event"] == 1
            else "No progression"
        ),

        "Concat Complete 30%": complete_dropout_risks[case_id]["missing_rna_30"]["Concat_fusion"],
        "Concat Missing 30%": missing_rna_risks[case_id]["missing_rna_30"]["Concat_fusion"],

        "Concat Complete 50%": complete_dropout_risks[case_id]["missing_rna_50"]["Concat_fusion"],
        "Concat Missing 50%": missing_rna_risks[case_id]["missing_rna_50"]["Concat_fusion"],

        "Gated Complete 30%": complete_dropout_risks[case_id]["missing_rna_30"]["gated_fusion"],
        "Gated Missing 30%": missing_rna_risks[case_id]["missing_rna_30"]["gated_fusion"],

        "Gated Complete 50%": complete_dropout_risks[case_id]["missing_rna_50"]["gated_fusion"],
        "Gated Missing 50%": missing_rna_risks[case_id]["missing_rna_50"]["gated_fusion"],

        "Time": labels_by_case.loc[case_id, "Time"],
    }
    for case_id in CASES
])

display(comparison_results)

,Patient,Outcome,Concat Complete 30%,Concat Missing 30%,Concat Complete 50%,Concat Missing 50%,Gated Complete 30%,Gated Missing 30%,Gated Complete 50%,Gated Missing 50%,Time
0,3A_186,Progression,4.407124,-4.554397,4.955022,-4.106859,4.909785,-3.760908,-4.848009,-4.875841,14
1,3B_418,Progression,0.306497,-1.558614,3.871257,4.081985,0.915986,-0.327534,0.099519,0.232236,12
2,3A_095,Progression,0.214300,0.747050,2.563612,2.632537,-0.935703,3.008761,2.159387,2.169758,6
3,3A_031,No progression,-4.720884,-4.669018,-5.292270,-5.276632,-4.992405,-4.964353,-5.014348,-5.014790,119
4,3A_169,No progression,-0.619974,-0.691901,-2.858013,-3.302879,-3.146972,-3.801513,-5.150784,-5.151448,155
5,3A_143,No progression,-4.713114,-4.686019,-5.317000,-5.314252,-4.977761,-4.965599,-5.037510,-5.037834,119
6,3B_426,Progression,0.047670,-1.531031,2.793151,3.328360,0.086574,-1.480586,-0.449945,-0.765127,5
7,3A_142,No progression,-0.748150,-0.952063,-3.531162,-3.607819,-4.751247,-4.535243,-5.103923,-5.104894,202
8,3B_385,No progression,0.200027,0.785925,2.574629,2.287537,-3.123755,3.196633,2.315992,2.321111,104
9,3A_041,Progression,-1.206232,-1.130555,3.954558,4.122061,0.599859,0.270651,0.401456,0.620603,10


In [ ]:
comparison_results = pd.DataFrame([
    {
        "Patient": case_id,
        "Outcome": (
            "Progression"
            if labels_by_case.loc[case_id, "Event"] == 1
            else "No progression"
        ),

        "Concat Complete 30%": complete_dropout_risks[case_id]["missing_rna_30"]["Concat_fusion"],
        "Concat Missing 30%": missing_rna_risks[case_id]["missing_rna_30"]["Concat_fusion"],

        "Concat Complete 50%": complete_dropout_risks[case_id]["missing_rna_50"]["Concat_fusion"],
        "Concat Missing 50%": missing_rna_risks[case_id]["missing_rna_50"]["Concat_fusion"],

        "Gated Complete 30%": complete_dropout_risks[case_id]["missing_rna_30"]["gated_fusion"],
        "Gated Missing 30%": missing_rna_risks[case_id]["missing_rna_30"]["gated_fusion"],

        "Gated Complete 50%": complete_dropout_risks[case_id]["missing_rna_50"]["gated_fusion"],
        "Gated Missing 50%": missing_rna_risks[case_id]["missing_rna_50"]["gated_fusion"],

        "Time": labels_by_case.loc[case_id, "Time"],
    }
    for case_id in CASES
])


def color_outcome(value):
    if value == "Progression":
        return "background-color: #f4cccc; color: #990000; font-weight: bold"
    elif value == "No progression":
        return "background-color: #d9ead3; color: #38761d; font-weight: bold"
    return ""


comparison_results.style.map(
    color_outcome,
    subset=["Outcome"]
)

,Patient,Outcome,Concat Complete 30%,Concat Missing 30%,Concat Complete 50%,Concat Missing 50%,Gated Complete 30%,Gated Missing 30%,Gated Complete 50%,Gated Missing 50%,Time
0,3A_186,Progression,4.407124,-4.554397,4.955022,-4.106859,4.909785,-3.760908,-4.848009,-4.875841,14
1,3B_418,Progression,0.306497,-1.558614,3.871257,4.081985,0.915986,-0.327534,0.099519,0.232236,12
2,3A_095,Progression,0.214300,0.747050,2.563612,2.632537,-0.935703,3.008761,2.159387,2.169758,6
3,3A_031,No progression,-4.720884,-4.669018,-5.292270,-5.276632,-4.992405,-4.964353,-5.014348,-5.014790,119
4,3A_169,No progression,-0.619974,-0.691901,-2.858013,-3.302879,-3.146972,-3.801513,-5.150784,-5.151448,155
5,3A_143,No progression,-4.713114,-4.686019,-5.317000,-5.314252,-4.977761,-4.965599,-5.037510,-5.037834,119
6,3B_426,Progression,0.047670,-1.531031,2.793151,3.328360,0.086574,-1.480586,-0.449945,-0.765127,5
7,3A_142,No progression,-0.748150,-0.952063,-3.531162,-3.607819,-4.751247,-4.535243,-5.103923,-5.104894,202
8,3B_385,No progression,0.200027,0.785925,2.574629,2.287537,-3.123755,3.196633,2.315992,2.321111,104
9,3A_041,Progression,-1.206232,-1.130555,3.954558,4.122061,0.599859,0.270651,0.401456,0.620603,10


## 10. Comparing Predictions Across Patients

[TODO: Visualize the predicted risk scores across patients.]

- Compare models across the same patients
- Identify differences in predicted risk
- Visualize the results